In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score


In [9]:
import os
os.getcwd()

'/Users/bradlarson'

In [11]:
import os
os.chdir("/Users/bradlarson/Desktop/Applied-Project")


In [13]:
df= pd.read_csv("V2_ML_Water_Usage_Dataset.csv")
df.head()

,Wafer Step,Year,Synthetic_Water_per_Wafer,Synthetic_Water_per_Wafer_noisy,Investment_Level,Wafer_Size,Water_Efficiency_Level,Percent_Reclaimed_Water,Percent_Municipal_Water,Reclamation_Tech_Level,Wafer_Intention,Investment_Efficiency,Water_Stress_Index,Composite_Risk_Score,Year_Squared
0,Cleaning,1,0.000032,0.000034,218.543054,300,Low,82.021890,17.978110,5,Reduced,43.708611,5393.433070,146.106567,1
1,Cleaning,2,0.000033,0.000032,477.821438,300,Low,64.720525,35.279475,4,Reduced,119.455360,10583.842650,140.616157,4
2,Cleaning,3,0.000034,0.000035,379.397274,300,Low,56.316925,43.683075,5,Advanced,75.879455,13104.922450,138.395078,9
3,Cleaning,4,0.000035,0.000035,319.396318,450,Medium,73.515460,26.484540,2,Advanced,159.698159,11918.043170,202.654638,16
4,Cleaning,5,0.000036,0.000035,120.208388,200,Medium,59.200136,40.799864,4,Current,30.052097,8159.972703,98.960041,25


In [15]:
df_encoded = df.copy()

# Categorical columns to encode
label_cols = ['Wafer Step', 'Water_Efficiency_Level', 'Wafer_Intention']
encoder_dict = {}

for col in label_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    encoder_dict[col] = le  # Save encoder in case you want to decode later


In [17]:
feature_cols = [
    'Investment_Level',
    'Wafer_Size',
    'Water_Efficiency_Level',
    'Percent_Reclaimed_Water',
    'Percent_Municipal_Water',
    'Reclamation_Tech_Level',
    'Wafer_Intention',
    'Investment_Efficiency',
    'Water_Stress_Index',
    'Composite_Risk_Score',
    'Year_Squared'
]


In [19]:
# Filter only 'Cleaning' steps using the encoded value
cleaning_step_val = encoder_dict['Wafer Step'].transform(['Cleaning'])[0]
cleaning_df = df_encoded[df_encoded['Wafer Step'] == cleaning_step_val]


In [21]:
X = cleaning_df[feature_cols]
y = cleaning_df['Synthetic_Water_per_Wafer_noisy']  # or try 'Synthetic_Water_per_Wafer' later


In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [25]:
model = RandomForestRegressor(n_estimators=300, max_depth=5, min_samples_split=10, random_state=42)
model.fit(X_train, y_train)


RandomForestRegressor(max_depth=5, min_samples_split=10, n_estimators=300,
                      random_state=42)

In [27]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.8f}")
print(f"R² Score: {r2:.3f}")


MAE: 0.00000484
R² Score: 0.947


In [31]:
trained_models = {}

for step_val in sorted(df_encoded['Wafer Step'].unique()):
    step_name = encoder_dict['Wafer Step'].inverse_transform([step_val])[0]
    step_df = df_encoded[df_encoded['Wafer Step'] == step_val]

    if len(step_df) < 10:
        continue

    X = step_df[feature_cols]
    y = step_df['Synthetic_Water_per_Wafer_noisy']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = RandomForestRegressor(n_estimators=300, max_depth=5, min_samples_split=10, random_state=42)
    model.fit(X_train, y_train)

    trained_models[step_name] = model


In [35]:
X_new = pd.DataFrame([{
    'Investment_Level': 300,
    'Wafer_Size': 300,
    'Water_Efficiency_Level': encoder_dict['Water_Efficiency_Level'].transform(['Medium'])[0],
    'Percent_Reclaimed_Water': 65,
    'Percent_Municipal_Water': 35,
    'Reclamation_Tech_Level': 3,
    'Wafer_Intention': encoder_dict['Wafer_Intention'].transform(['Advanced'])[0],
    'Investment_Efficiency': 100.0,  # e.g., Investment_Level / Tech_Level
    'Water_Stress_Index': 300 * 35,  # Wafer Size * Municipal %
    'Composite_Risk_Score': 300 * 0.4 + 3 * 0.3 + 65 * 0.3,
    'Year_Squared': 25 ** 2
}])


In [37]:
prediction = trained_models['Cleaning'].predict(X_new)[0]
print(f"Predicted Water Use per Wafer: {prediction:.8f} liters")


Predicted Water Use per Wafer: 0.00006542 liters


In [39]:
prediction = trained_models['CMP'].predict(X_new)[0]
print(f"Predicted Water Use per Wafer: {prediction:.8f} liters")


Predicted Water Use per Wafer: 0.00003205 liters


In [43]:
# Function to evaluate model performance by wafer step
def evaluate_models_by_step(df, encoder_dict, feature_cols, target_col='Synthetic_Water_per_Wafer_noisy'):
    results = []
    trained_models = {}

    wafer_steps = sorted(df['Wafer Step'].unique())

    for step_val in wafer_steps:
        step_name = encoder_dict['Wafer Step'].inverse_transform([step_val])[0]
        step_df = df[df['Wafer Step'] == step_val]

        if len(step_df) < 10:
            results.append((step_name, len(step_df), "Too few rows", None))
            continue

        X = step_df[feature_cols]
        y = step_df[target_col]

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        model = RandomForestRegressor(n_estimators=300, max_depth=5, min_samples_split=10, random_state=42)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        trained_models[step_name] = model
        results.append((step_name, len(step_df), round(mae, 8), round(r2, 3)))

    results_df = pd.DataFrame(results, columns=["Wafer Step", "Samples", "MAE", "R² Score"])
    return results_df, trained_models

# ✅ Run the function with your data
results_df, trained_models = evaluate_models_by_step(df_encoded, encoder_dict, feature_cols)

# ✅ Display the table in your notebook
results_df

,Wafer Step,Samples,MAE,R² Score
0,CMP,50,2.100000e-06,0.945
1,Cleaning,50,4.840000e-06,0.947
2,Deposition,50,1.360000e-06,0.937
3,Doping,50,6.600000e-07,0.966
4,Etching,50,1.480000e-06,0.962
5,Inspection,50,7.900000e-07,0.949
6,Oxidation,50,9.900000e-07,0.933
7,Packaging,50,7.500000e-07,0.943
8,Photolithography,50,3.010000e-06,0.937
